In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# This script processes the VX data and creates a long-type dataframe.
#  It reads VX data from a CSV file, transforms it into a long format,
#  and saves the result to a new CSV file.
# 26.01.11 문장 끝 어미에 동사가 없는 경우에 대해서 수정한 파일.

import pandas as pd
import numpy as np
from typing import Tuple, List

import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))
from stats.filtering import apply_filters, FilterValue, has_value
from stats.vx_make import make_annotate_vx #EN_No 수정
from stats.en_no_fix import fix_en_number_with_merge, make_en_no_sub #EN_No 수정
from stats.v_affix_attach import v_affix_attach, add_v_no_by_merge_and_fallback

In [ ]:
from pathlib import Path
from datetime import datetime

#대상 파일 읽어오기.
CSV_PATH = Path(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.2.csv")#"..\csv\_patched\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9__patched__20260110_024723.csv"
df = pd.read_csv(CSV_PATH)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

In [ ]:
# 1. CSV → df
OUT_DIR = Path(r"..\csv\_patched")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ✅ 출력 파일 및 폴더

OUT_CSV = OUT_DIR / f"{CSV_PATH.stem}_후처리.csv"
df.columns

In [ ]:
# 후처리 1. VX 만들기. 
# stats.vx_make의 함수 이용(C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\stats\vx_make.py)
KEEP_COLS = [
    "ID", "file_id", "category", "sen_id", "word_id2", "form/label",
    "N_form", "N_label", "V_form", "V_label",
    "EP_form", "EP_label", "EN_form", "EN_label", "J_form", "J_label",
    "EN_No"
]
VX_LABEL_PATH = r"..\..\label\000.vx_label(2026)_3.csv"
EDGES_FILE = OUT_DIR / "patch_info"/ f"{CSV_PATH.stem}_edges.csv"
# 원본 유지 열만 사용
df_base = df[[c for c in KEEP_COLS if c in df.columns]].copy()
df_out, edges = make_annotate_vx(df_base, VX_LABEL_PATH) #stats.vx_make

edges.to_csv(EDGES_FILE, index=False, encoding="utf-8-sig")


In [ ]:
df_out.head(1000)

In [ ]:
# 후처리2 EN_No 결번 채움.
EN_LABEL_PATH = r"..\..\label\000.en_labels2.csv"

df_out["EN_No"] = np.floor(df["EN_No"]).astype("Int64") #소수점일 때 내림.
df_out["EN_No"] = pd.to_numeric(df["EN_No"], errors="coerce").astype("Int64") #float 등일 때 Int64로 바꿈.
df_out = fix_en_number_with_merge(df_out, EN_LABEL_PATH, "EN", False)

In [ ]:
df_out.columns

In [ ]:
df.loc[has_value(df["EN_form"])&~has_value(df["EN_No"]), "EN_form"]


In [ ]:
df_out.loc[has_value(df["EN_form"])&~has_value(df["EN_No"]), "EN_No"]


In [ ]:
# 후처리3 EN_No_sub 만듦.
df_out = make_en_no_sub(df_out)
df_out.columns

In [ ]:
### 후처리4 category 변환
df = df_out.copy()

# 누락값 채움 & category 변환
#%% 누락값 채움. 
#object를 category형으로 바꿈. : 45초 걸림

exclude_cols = ['form/label', 'N_form', 'V_form']  # 제외할 열

for col in df.columns:
    if col in exclude_cols:
        continue  # 제외할 열은 건너뛰기

    # 숫자형이면 -1로 채움
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(-1)

    # 문자열(object)이면 ''로 채우고 고유값 적을 때만 category로
    elif pd.api.types.is_object_dtype(df[col]):
        df[col] = df[col].fillna("")
        unique_ratio = df[col].nunique(dropna=False) / len(df)
        if unique_ratio < 0.5:
            df[col] = df[col].astype('category')

# EP정보 덧붙이기
if True:
    # --- EP_form & flags ---
    if 'EP_form' in df.columns:
        s = df['EP_form'].astype('string').str.replace(' + ', '', regex=False)
        df['EP_form'] = s

#category 업데이트
# --- file_id → new category 적용 --- 
if True:
    mapping_df = pd.read_csv('../label/TAG_file_Id_new_category.csv')
    id_to_category = dict(zip(mapping_df['file_id'], mapping_df['category']))

    # --- 1. docu_id → file_id로 통일 ---
    if 'file_id' not in df.columns and 'docu_id' in df.columns:
        df = df.rename(columns={'docu_id': 'file_id'})

    # --- 2. category 열이 없으면 생성 ---
    if 'category' not in df.columns:
        df['category'] = pd.NA

    # --- 3. file_id 기준으로 category 매핑 ---
    if 'file_id' in df.columns:
        df['category'] = (
            df['file_id']
            .map(id_to_category)
            .fillna(df['category'])
        )


In [ ]:
#후처리5 - 문장 끝 V 표시(V_label 또는 EN_label 기준)

import pandas as pd
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))
from stats.filtering import has_value

def has_value(s: pd.Series) -> pd.Series:
    """NaN / 빈문자열 / 'NULL' 제외한 값 여부"""
    return (
        s.notna()
        & s.astype(str).str.strip().ne("")
        & s.astype(str).str.upper().ne("NULL")
    )

# 1) V 여부 마스크: V_label OR EN_label
v_mask = has_value(df["V_label"]) | has_value(df["EN_label"])

# 2) sen_id별 마지막 V 인덱스
last_v_idx = df.loc[v_mask].groupby("sen_id").tail(1).index

# 3) 표시 컬럼
df["sent_end_V"] = False
df.loc[last_v_idx, "sent_end_V"] = True

# 4) sentence-end (always one per sen_id)
last_row_idx = df.groupby("sen_id").tail(1).index

df["sent_end"] = False
df.loc[last_row_idx, "sent_end"] = True


In [ ]:
# XSV, XSA 붙여서 동사열 새로 만들기
df_out = v_affix_attach(df)

In [ ]:
#V_No 붙이기
df_out = df[['ID', 'file_id', 'category', 'sen_id', 'word_id2', 'form/label',
       'N_form', 'N_label', 'V_form', 'V_label', 'EP_form', 'EP_label',
       'EN_form', 'EN_label', 'J_form', 'J_label', 'EN_No', 'Next_VX_No',
       'Next_VX_No_form', 'vx0_No', 'vx0_form', 'vx0_order', 'vx_len',
       'V_word_id', 'f_word_id', 'EN_No_sub', 'sent_end_V', 'V_form_0',
       'V_label_0']]
       
V_No_file = r"..\..\label\000.V_No_(2026).csv"
df_out = add_v_no_by_merge_and_fallback(df_out, V_No_file, encoding='utf-8-sig') #"cp949"

In [ ]:
df_out.to_csv(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.2.csv", index=False, encoding="utf-8-sig")